# Chapitre 6 : Caractérisation des exploitations et de l'adoption agroécologique

**Auteur :** Ibrahima Baldé, Master 2 ASAD, USSEIN (Kaolack, Sénégal)
**Lien avec le mémoire :** Chapitre 6 (sections 6.1 à 6.4)
**Entrée :** `data/Base_Enquete_Kaolack_178.xlsx` (non publiée, voir README)
**Contenu :** profil sociodémographique, foncier et accès aux services, validation de l'IAA (25 items), typologie (ACP + CAH), diversité alimentaire HDDS et WDDS.
**Exécution :** lancer les cellules dans l'ordre, de haut en bas.

# Analyse des données d'enquetes de terrain dans les zone urbain et periurbain de la comune de kaolack

## IMPORTATION DES BIBLIOTHÈQUES ET CHARGEMENT DE LA BASE DE DONNÉES

In [ ]:
# 1. Bibliothèques standard et traitement de données
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# 2. Statistiques et modélisation
import pingouin as pg
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
# 3. Machine Learning & Reduction de dimension (ACP / CAH)
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# Configuration du style graphique pour des figures académiques (300 DPI)
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

In [ ]:
# Création automatique des dossiers de sortie pour les tableaux et graphiques
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
os.makedirs("tables", exist_ok=True)
os.makedirs("figures", exist_ok=True)

print("✅ Bibliothèques importées et dossier de sortie configurés avec succès.")

In [ ]:
### ==============================================================================
# 4. Chargement de la base de données et Vérifications
# ==============================================================================
import pandas as pd
# Nom et chemin complet du fichier Excel
file_path = "data/Base_Enquete_Kaolack_178.xlsx"

# Chargement dans un DataFrame pandas
df = pd.read_excel(file_path)

In [ ]:
# ------------------------------------------------------------------------------
# Contrôles de validation (Validation des 178 ménages)
# ------------------------------------------------------------------------------
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
print("\n" + "=" * 50)
print("VERIFICATION INITIALE DE LA BASE")
print("=" * 50)

# Vérification du nombre d'observations et de variables
lignes, colonnes = df.shape
print(f"-> Nombre de ménages enregistrés (lignes) : {lignes}")
print(f"-> Nombre de variables (colonnes)         : {colonnes}")

In [ ]:
# Test de validation sur l'échantillon N = 178
if lignes == 178:
    print(
        "✅ VALIDATION : Le nombre d'observations correspond parfaitement à l'échantillon cible (N = 178)."
    )
else:
    print(
        f"⚠️ ATTENTION : La base contient {lignes} lignes au lieu des 178 attendues."
    )

In [ ]:
# Aperçu des 5 premières lignes
print("\n--- Aperçu des 5 premières lignes ---")
display(df.head())

# Structure des types de données et valeurs manquantes principales
print("\n--- Informations sur les colonnes et valeurs manquantes ---")
print(
    f"Total de valeurs manquantes dans la base : {df.isnull().sum().sum()}"
)

## Section 6.1 : Caractérisation Socio-Économique, Foncière et Institutionnelle

## 6.1.1 Profil sociodémographique et capital humain

#### Objectif académique : Établir le profil de référence des ménages et tester l'homogénéité/hétérogénéité entre la zone urbaine (Ngane Alassane,Parcelles Assainies, Sam Moussa, Kabatoki) et la zone périurbaine (Kahone, Sibassor, Ndiaffate, Mbadakhoune, Khelcom Birane).

In [ ]:
for i, col in enumerate(df.columns):
    print(f"[{i}] {col}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display  # Pour l'affichage du tableau dans l'interface

# =============================================================================
# 1. DÉTECTION AUTOMATIQUE ET FIABLE DE LA ZONE (89 URBAIN / 89 PÉRIURBAIN)
# =============================================================================
col_id = None
for col in df.columns:
    matches = df[col].astype(str).str.strip().str.upper().str.startswith(('U-', 'P-'))
    if matches.sum() > 50:
        col_id = col
        break

if col_id is not None:
    df['zone_etude'] = df[col_id].astype(str).str.strip().str.upper().apply(
        lambda x: 'Urbain' if x.startswith('U') else ('Périurbain' if x.startswith('P') else np.nan)
    )
    df['zone'] = df['zone_etude']
    print(f"--> Identifiant trouvé : '{col_id}'")
else:
    print("ATTENTION : Colonne d'identifiant U-/P- non trouvée.")

print("\nEffectifs validés par zone d'étude :")
print(df['zone_etude'].value_counts(dropna=False))

# =============================================================================
# 2. NETTOYAGE ET RECODAGE SÉCURISÉ DES DONNÉES
# =============================================================================
mapping_niveau = {
    0: 'Aucun / Non scolarisé', '0': 'Aucun / Non scolarisé',
    1: 'Primaire', '1': 'Primaire',
    2: 'Secondaire', '2': 'Secondaire',
    3: 'Supérieur', '3': 'Supérieur'
}

mapping_statut_matri = {
    1: 'Marié(e)', '1': 'Marié(e)',
    2: 'Célibataire', '2': 'Célibataire',
    3: 'Veuf / Veuve', '3': 'Veuf / Veuve',
    4: 'Divorcé(e)', '4': 'Divorcé(e)'
}

mapping_habitat = {
    1: 'Logement en dur (béton, villa, immeuble)', '1': 'Logement en dur (béton, villa, immeuble)',
    2: 'Logement en semi-dur (briques de ciment, toit en tôle)', '2': 'Logement en semi-dur (briques de ciment, toit en tôle)',
    3: 'Logement traditionnel / précaire (banco, paille, bois)', '3': 'Logement traditionnel / précaire (banco, paille, bois)',
    4: 'Autre', '4': 'Autre'
}

mapping_statut_foncier = {
    'heritage': 'Héritage familial (propriété coutumière traditionnelle)',
    'affectation_commune': 'Affectation par la Mairie / Conseil municipal (titre officiel)',
    'location': 'Location ou métayage (paiement en argent ou part de récolte)',
    'pret_gratuit': 'Prêt gratuit par un tiers ou un parent',
    'achat': 'Achat privé',
    1: 'Affectation par la Mairie / Conseil municipal (titre officiel)',
    '1': 'Affectation par la Mairie / Conseil municipal (titre officiel)'
}

# Application des nettoyages
df[df.columns[16]] = df[df.columns[16]].replace(mapping_niveau)
df[df.columns[17]] = df[df.columns[17]].replace(mapping_statut_matri)
df[df.columns[20]] = df[df.columns[20]].replace(mapping_habitat)
df[df.columns[21]] = df[df.columns[21]].replace(mapping_statut_foncier)

# =============================================================================
# 3. FONCTION D'ANALYSE COMPARATIVE (ORDRE STRICT URBAIN -> PÉRIURBAIN)
# =============================================================================
def analyser_profil_sociodemo(df, col_zone, vars_continues, vars_categorielles, output_filename="Tableau_6_1_1_Sociodemo_Comparatif.xlsx"):
    df_anal = df.dropna(subset=[col_zone]).copy()
    
    # FORÇAGE STRICT DE L'ORDRE DES ZONES
    zone1, zone2 = 'Urbain', 'Périurbain'
    resultats = []

    # --- Variables continues ---
    for var in vars_continues:
        g1 = df_anal[df_anal[col_zone] == zone1][var].dropna()
        g2 = df_anal[df_anal[col_zone] == zone2][var].dropna()
        
        if len(g1) < 3 or len(g2) < 3:
            continue
            
        m1, sd1 = g1.mean(), g1.std()
        m2, sd2 = g2.mean(), g2.std()

        p_shapiro1 = stats.shapiro(g1).pvalue if len(g1) >= 3 else 0
        p_shapiro2 = stats.shapiro(g2).pvalue if len(g2) >= 3 else 0
        is_normal = (p_shapiro1 > 0.05) and (p_shapiro2 > 0.05)
        
        if is_normal:
            test_nom = "t-test de Student"
            stat, p_val = stats.ttest_ind(g1, g2, equal_var=True)
        else:
            test_nom = "Mann-Whitney U"
            stat, p_val = stats.mannwhitneyu(g1, g2, alternative='two-sided')
            
        p_str = "< 0.001" if p_val < 0.001 else f"{p_val:.4f}"
        
        resultats.append({
            'Variable': var,
            'Modalité / Indicateur': 'Moyenne ± Écart-type',
            f'{zone1}': f"{m1:.2f} ± {sd1:.2f}",
            f'{zone2}': f"{m2:.2f} ± {sd2:.2f}",
            'Test appliqué': test_nom,
            'Statistique': round(stat, 3),
            'p-value': p_str
        })

    # --- Variables catégorielles ---
    for var in vars_categorielles:
        contingency = pd.crosstab(df_anal[var], df_anal[col_zone])
        proportions = pd.crosstab(df_anal[var], df_anal[col_zone], normalize='columns') * 100
        
        chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)
        has_low_expected = (expected < 5).any()
        
        if has_low_expected and contingency.shape == (2, 2):
            test_nom = "Fisher Exact"
            stat, p_val = stats.fisher_exact(contingency)
        else:
            test_nom = "Chi-deux (χ²)"
            stat, p_val = chi2, p_chi2
            
        p_str = "< 0.001" if p_val < 0.001 else f"{p_val:.4f}"
        
        for i, mod in enumerate(contingency.index):
            eff1 = contingency.loc[mod, zone1] if zone1 in contingency.columns else 0
            prop1 = proportions.loc[mod, zone1] if zone1 in proportions.columns else 0.0
            eff2 = contingency.loc[mod, zone2] if zone2 in contingency.columns else 0
            prop2 = proportions.loc[mod, zone2] if zone2 in proportions.columns else 0.0
            
            resultats.append({
                'Variable': var if i == 0 else "",
                'Modalité / Indicateur': str(mod),
                f'{zone1}': f"{eff1} ({prop1:.1f}%)",
                f'{zone2}': f"{eff2} ({prop2:.1f}%)",
                'Test appliqué': test_nom if i == 0 else "",
                'Statistique': round(stat, 3) if i == 0 else "",
                'p-value': p_str if i == 0 else ""
            })

    df_export = pd.DataFrame(resultats)
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        df_export.to_excel(writer, sheet_name='Sociodemo_Comparatif', index=False)
        
    print(f"\n--> Tableau Excel sauvegardé : {output_filename}")
    return df_export

# =============================================================================
# 4. EXÉCUTION DU CALCUL ET AFFICHAGE DU TABLEAU DANS L'INTERFACE
# =============================================================================
mes_vars_continues = [
    df.columns[14],  # Âge du chef de ménage
    df.columns[18],  # Nombre de personnes dans le ménage
    df.columns[19]   # Enfants de moins de 5 ans
]

mes_vars_categorielles = [
    df.columns[15],  # Sexe du chef de ménage
    df.columns[16],  # Niveau d'instruction
    df.columns[17],  # Statut matrimonial
    df.columns[20],  # Type d'habitat
    df.columns[21]   # Statut foncier
]

df_resultats = analyser_profil_sociodemo(
    df=df,
    col_zone='zone_etude',
    vars_continues=mes_vars_continues,
    vars_categorielles=mes_vars_categorielles,
    output_filename="Tableau_6_1_1_Sociodemo_Comparatif.xlsx"
)

# AFFICHAGE DU TABLEAU SUR L'INTERFACE JUPYTERLAB
display(df_resultats)

# =============================================================================
# 5. GÉNÉRATION DE LA FIGURE 6.1 HARMONISÉE
# =============================================================================
sns.set_theme(style="whitegrid", font_scale=1.0)
couleurs_zone = {'Urbain': '#1f77b4', 'Périurbain': '#ff7f0e'}
ordre_zones = ['Urbain', 'Périurbain']

plt.close('all')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Graphique A : Instruction
col_instruction = df.columns[16]
prop_inst = pd.crosstab(df[col_instruction], df['zone_etude'], normalize='columns') * 100
df_inst = prop_inst.reset_index().melt(id_vars=col_instruction, value_name='Pourcentage', var_name='Zone')

sns.barplot(data=df_inst, x=col_instruction, y='Pourcentage', hue='Zone', hue_order=ordre_zones, palette=couleurs_zone, ax=axes[0, 0])
axes[0, 0].set_title("A. Niveau d'instruction du chef de ménage (%)", fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel("Niveau d'instruction")
axes[0, 0].set_ylabel("Pourcentage (%)")
axes[0, 0].tick_params(axis='x', rotation=20)
for p in axes[0, 0].patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        axes[0, 0].annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2., h),
                            ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')

# Graphique B : Foncier
col_foncier = df.columns[21]
prop_foncier = pd.crosstab(df[col_foncier], df['zone_etude'], normalize='columns') * 100
df_foncier = prop_foncier.reset_index().melt(id_vars=col_foncier, value_name='Pourcentage', var_name='Zone')

sns.barplot(data=df_foncier, x=col_foncier, y='Pourcentage', hue='Zone', hue_order=ordre_zones, palette=couleurs_zone, ax=axes[0, 1])
axes[0, 1].set_title("B. Statut foncier de l'exploitation (%)", fontweight='bold', fontsize=12)
axes[0, 1].set_xlabel("Statut foncier")
axes[0, 1].set_ylabel("Pourcentage (%)")
axes[0, 1].tick_params(axis='x', rotation=25)
for p in axes[0, 1].patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        axes[0, 1].annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2., h),
                            ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')

# Graphique C : Taille du ménage
col_taille = df.columns[18]
sns.boxplot(data=df, x='zone_etude', y=col_taille, order=ordre_zones, hue='zone_etude', palette=couleurs_zone, ax=axes[1, 0], width=0.4, showmeans=True,
            meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
axes[1, 0].set_title("C. Distribution de la taille du ménage (personnes)", fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel("Zone d'étude")
axes[1, 0].set_ylabel("Nombre de personnes par ménage")

# Graphique D : Type d'habitat
col_habitat = df.columns[20]
prop_habitat = pd.crosstab(df[col_habitat], df['zone_etude'], normalize='columns') * 100
df_habitat = prop_habitat.reset_index().melt(id_vars=col_habitat, value_name='Pourcentage', var_name='Zone')

sns.barplot(data=df_habitat, x=col_habitat, y='Pourcentage', hue='Zone', hue_order=ordre_zones, palette=couleurs_zone, ax=axes[1, 1])
axes[1, 1].set_title("D. Type d'habitat du ménage (%)", fontweight='bold', fontsize=12)
axes[1, 1].set_xlabel("Type d'habitat")
axes[1, 1].set_ylabel("Pourcentage (%)")
axes[1, 1].tick_params(axis='x', rotation=20)
for p in axes[1, 1].patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        axes[1, 1].annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2., h),
                            ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')

plt.suptitle("Fig 6.1 : Profil sociodémographique comparatif des ménages (Urbain n=89 vs Périurbain n=89)", 
             fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
fichier_fig61 = "Fig_6_1_Profil_Sociodemo_Comparatif.png"
plt.savefig(fichier_fig61, dpi=300, bbox_inches='tight')
plt.show()

print(f"--> Figure sauvegardée en haute définition : {fichier_fig61}")

## 6.1.2 Structure foncière, accès aux ressources et contraintes salines


#### 1.1. Analyse des variables continues (Âge, Taille ménage, Superficie, Actifs)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from IPython.display import display

# ==============================================================================
# 1. FONCTION : Calcul de la médiane et IQR
# ==============================================================================
def calculer_indicateurs_fonciers(df, col_zone='zone', col_superficie=None):
    """
    Calcule la médiane, Q1, Q3 et l'IQR par zone pour une variable de superficie.
    """
    if col_superficie is None:
        col_superficie = df.columns[481]
        
    def stats_non_parametriques(x):
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        med = x.median()
        iqr = q3 - q1
        return pd.Series({
            'Médiane (ha)': round(med, 2),
            'Q1_25% (ha)': round(q1, 2),
            'Q3_75% (ha)': round(q3, 2),
            'IQR (ha)': round(iqr, 2)
        })
    
    return df.groupby(col_zone)[col_superficie].apply(stats_non_parametriques).unstack().reset_index()

# ==============================================================================
# 2. FONCTION : Cross-tabulation et Test du Chi2
# ==============================================================================
def analyser_tableau_croise(df, col_zone='zone', col_variable='statut_foncier'):
    """
    Génère la table de contingence et effectue le test d'indépendance du Chi2.
    """
    contingency_table = pd.crosstab(df[col_zone], df[col_variable], margins=True, margins_name='Total')
    contingency_clean = pd.crosstab(df[col_zone], df[col_variable])
    
    chi2, p_val, dof, expected = chi2_contingency(contingency_clean)
    
    stats_chi2 = {
        'Variable': col_variable,
        'Statistique Chi2': round(chi2, 3),
        'Degrés de liberté (ddl)': dof,
        'p-value': round(p_val, 4),
        'Significatif (p < 0.05)': p_val < 0.05
    }
    
    return contingency_table, pd.DataFrame([stats_chi2])

# ==============================================================================
# 3. FONCTION : Exportation des livrables Excel
# ==============================================================================
def exporter_livrable_excel(df, col_sup_totale=None, col_statut='statut_foncier', col_salinite='score_salinite', filename='Tableau_6_1_2_Foncier_Salinite.xlsx'):
    """
    Exporte l'ensemble des résultats de l'analyse foncière et de salinité dans un fichier Excel multi-feuilles.
    """
    if col_sup_totale is None:
        col_sup_totale = df.columns[481]
        
    stats_foncier = calculer_indicateurs_fonciers(df, col_zone='zone', col_superficie=col_sup_totale)
    ct_statut, chi2_statut = analyser_tableau_croise(df, col_zone='zone', col_variable=col_statut)
    ct_salinite, chi2_salinite = analyser_tableau_croise(df, col_zone='zone', col_variable=col_salinite)
    
    df_chi2_summary = pd.concat([chi2_statut, chi2_salinite], ignore_index=True)
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        stats_foncier.to_excel(writer, sheet_name='Superficie_Médianes_IQR', index=False)
        ct_statut.to_excel(writer, sheet_name='Statut_Foncier_Zone')
        ct_salinite.to_excel(writer, sheet_name='Salinite_Zone')
        df_chi2_summary.to_excel(writer, sheet_name='Tests_Chi2_Synthese', index=False)
        
    print(f"✅ Fichier Excel exporté avec succès : {filename}")

# ==============================================================================
# 4. FONCTION : Génération de la Figure 6.1
# ==============================================================================
def generer_figure_6_1(df, col_statut='statut_foncier', col_salinite='score_salinite', output_img='Fig_6_1_Contraintes_Foncières_Salinite.png'):
    """
    Génère la figure 6.1 avec une disposition optimisée sans chevauchement des légendes.
    """
    sns.set_theme(style='whitegrid', font='DejaVu Sans')
    
    # Utilisation de layout='constrained' au lieu de tight_layout pour gérer l'espace des légendes
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), layout='constrained')

    # --- Graphique A : Statut Foncier ---
    pct_statut = pd.crosstab(df['zone'], df[col_statut], normalize='index') * 100
    pct_statut.plot(kind='bar', stacked=True, ax=axes[0], colormap='Blues', edgecolor='black', linewidth=0.8)
    axes[0].set_title('A. Mode de faire-valoir foncier par zone', fontsize=12, fontweight='bold', pad=10)
    axes[0].set_ylabel('Proportion (%)', fontsize=11)
    axes[0].set_xlabel('Zone', fontsize=11)
    axes[0].legend(title='Statut Foncier', bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=8, frameon=True)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

    for p in axes[0].patches:
        height = p.get_height()
        if height > 4:
            x, y = p.get_xy()
            axes[0].text(x + p.get_width()/2, y + height/2, f'{height:.1f}%', 
                         ha='center', va='center', fontsize=9, fontweight='bold', color='black')

    # --- Graphique B : Score de Salinité ---
    pct_salinite = pd.crosstab(df['zone'], df[col_salinite], normalize='index') * 100
    pct_salinite.plot(kind='bar', stacked=True, ax=axes[1], colormap='YlOrRd', edgecolor='black', linewidth=0.8)
    axes[1].set_title('B. Degré de salinité perçue par zone', fontsize=12, fontweight='bold', pad=10)
    axes[1].set_ylabel('Proportion (%)', fontsize=11)
    axes[1].set_xlabel('Zone', fontsize=11)
    axes[1].legend(title='Degré de Salinité', bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=8, frameon=True)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

    for p in axes[1].patches:
        height = p.get_height()
        if height > 4:
            x, y = p.get_xy()
            axes[1].text(x + p.get_width()/2, y + height/2, f'{height:.1f}%', 
                         ha='center', va='center', fontsize=9, fontweight='bold', color='black')

    plt.savefig(output_img, dpi=300, bbox_inches='tight')
    plt.show()

# ==============================================================================
# 5. SCRIPT D'EXÉCUTION PRINCIPAL (CORRIGÉ)
# ==============================================================================

# 1. Normalisation de la zone
if 'zone' not in df.columns and 'zone_etude' in df.columns:
    df['zone'] = df['zone_etude']

# 2. Assignation dynamique des colonnes par leur nom ou leur index exact
col_sup_totale   = df.columns[481]
col_sup_agroeco  = df.columns[482]
col_taux_emprise = df.columns[483]

# Remplacez les chiffres ci-dessous par les index trouvés à l'étape 1 :
col_statut_foncier = df.columns[484]  # <-- Remplacez 484 par l'index du statut foncier
col_salinite       = df.columns[486]  # <-- Remplacez 485 par l'index de la salinité

# 3. Sécurisation du typage numérique
for col in [col_sup_totale, col_sup_agroeco, col_taux_emprise]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Affichage des statistiques de superficie
print("=== 1. SUPERFICIE TOTALE CULTIVÉE (HA) ===")
display(calculer_indicateurs_fonciers(df, col_zone='zone', col_superficie=col_sup_totale))

print("\n=== 2. SUPERFICIE CONSACRÉE À L'AGROÉCOLOGIE (HA) ===")
display(calculer_indicateurs_fonciers(df, col_zone='zone', col_superficie=col_sup_agroeco))

print("\n=== 3. TAUX D'EMPRISE AGROÉCOLOGIQUE (%) ===")
display(df.groupby('zone')[col_taux_emprise].describe().round(2))

# 5. Tables croisées et Chi² avec les colonnes corrigées
table_statut, chi2_statut = analyser_tableau_croise(df, col_zone='zone', col_variable=col_statut_foncier)
table_salinite, chi2_salinite = analyser_tableau_croise(df, col_zone='zone', col_variable=col_salinite)

print("\n=== 4. STATUT FONCIER PAR ZONE ===")
display(table_statut)
display(chi2_statut)

print("\n=== 5. DEGRÉ DE SALINITÉ PAR ZONE ===")
display(table_salinite)
display(chi2_salinite)

# 6. Exportation Excel et Figure
exporter_livrable_excel(
    df, 
    col_sup_totale=col_sup_totale, 
    col_statut=col_statut_foncier, 
    col_salinite=col_salinite
)

generer_figure_6_1(
    df, 
    col_statut=col_statut_foncier, 
    col_salinite=col_salinite
)
# 1. Vérification des effectifs bruts par zone
print("--- Effectifs bruts dans la colonne zone ---")
print(df['zone'].value_counts(dropna=False))

### 6.1.3 Environnement institutionnel, financier et services agricoles

#### Objectif académique : Évaluer l'intégration des producteurs dans les réseaux d'accompagnement et de crédit.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from IPython.display import display

# ==============================================================================
# 1. ANALYSE DES VARIABLES BINAIRES (ACCÈS AUX SERVICES, CRÉDIT, OP, FORMATION)
# ==============================================================================
def analyser_services_binaires(df, col_zone='zone', cols_binaires=None):
    """
    Calcule les taux de pénétration (% 'Oui') globaux et par zone,
    puis effectue un test d'indépendance du Chi2 (Urbain vs. Périurbain).
    """
    if cols_binaires is None:
        cols_binaires = ['credit_acces', 'adhesion_op', 'acces_vulgarisation', 'formation_agroeco']
        
    # Détection automatique des libellés exacts des zones dans le dataframe
    zones_uniques = df[col_zone].astype(str).unique()
    zone_urb = next((z for z in zones_uniques if 'urbain' in z.lower() and 'péri' not in z.lower() and 'peri' not in z.lower()), 'Urbain')
    zone_peri = next((z for z in zones_uniques if 'péri' in z.lower() or 'peri' in z.lower()), 'Périurbain')

    resultats = []
    for col in cols_binaires:
        if col not in df.columns:
            print(f"⚠️ Variable non trouvée dans df : {col}")
            continue

        # Nettoyage et normalisation des modalités
        s_clean = df[col].astype(str).str.strip().str.capitalize()
        
        # Effectifs et pourcentages globaux
        total_oui = (s_clean == 'Oui').sum()
        pct_global = (total_oui / len(df)) * 100
        
        # Table de contingence et pourcentages par zone
        ct = pd.crosstab(df[col_zone], s_clean)
        pct_zone = pd.crosstab(df[col_zone], s_clean, normalize='index') * 100
        
        pct_urbain = pct_zone.loc[zone_urb, 'Oui'] if ('Oui' in pct_zone.columns and zone_urb in pct_zone.index) else 0.0
        pct_periurbain = pct_zone.loc[zone_peri, 'Oui'] if ('Oui' in pct_zone.columns and zone_peri in pct_zone.index) else 0.0
        
        # Test du Chi2 sur la table de contingence
        chi2, p_val, dof, _ = chi2_contingency(ct)
        
        resultats.append({
            'Indicateur / Service': col,
            'Effectif Réponse (Oui)': total_oui,
            'Taux Global (%)': round(pct_global, 2),
            'Taux Urbain (%)': round(pct_urbain, 2),
            'Taux Périurbain (%)': round(pct_periurbain, 2),
            'Statistique Chi2': round(chi2, 3),
            'ddl': dof,
            'p-value': round(p_val, 4),
            'Significatif (p < 0.05)': 'Oui' if p_val < 0.05 else 'Non'
        })
        
    return pd.DataFrame(resultats)

# ==============================================================================
# 2. TRAITEMENT DES RÉPONSES MULTIPLES (ONE-HOT ENCODING) : PRESTATAIRES
# ==============================================================================
def analyser_reponses_multiples(df, col_multiple='prestataire_type', col_zone='zone', sep=';'):
    """
    Effectue l'éclatement binaire pour une variable à choix multiples
    et calcule les taux de citation par zone avec test Chi2.
    """
    if col_multiple not in df.columns:
        print(f"⚠️ Variable à choix multiples non trouvée : {col_multiple}")
        return pd.DataFrame()

    # Détection automatique des zones
    zones_uniques = df[col_zone].astype(str).unique()
    zone_urb = next((z for z in zones_uniques if 'urbain' in z.lower() and 'péri' not in z.lower() and 'peri' not in z.lower()), 'Urbain')
    zone_peri = next((z for z in zones_uniques if 'péri' in z.lower() or 'peri' in z.lower()), 'Périurbain')

    # Éclatement binaire des réponses multiples
    df_dummies = df[col_multiple].fillna('').astype(str).str.get_dummies(sep=sep)
    df_dummies[col_zone] = df[col_zone]
    
    prestataires = [c for c in df_dummies.columns if c != col_zone]
    resultats_multiples = []
    
    n_total = len(df)
    n_urbain = (df[col_zone] == zone_urb).sum()
    n_periurbain = (df[col_zone] == zone_peri).sum()
    
    for pres in prestataires:
        cnt_total = df_dummies[pres].sum()
        pct_global = (cnt_total / n_total) * 100
        
        cnt_urbain = df_dummies[df_dummies[col_zone] == zone_urb][pres].sum()
        cnt_periurbain = df_dummies[df_dummies[col_zone] == zone_peri][pres].sum()
        
        pct_urbain = (cnt_urbain / n_urbain) * 100 if n_urbain > 0 else 0.0
        pct_periurbain = (cnt_periurbain / n_periurbain) * 100 if n_periurbain > 0 else 0.0
        
        # Test du Chi2
        ct_p = pd.crosstab(df_dummies[col_zone], df_dummies[pres])
        chi2, p_val, dof, _ = chi2_contingency(ct_p)
        
        resultats_multiples.append({
            'Type de Prestataire / Modalité': pres,
            'Citations Totales': cnt_total,
            'Taux Citation Global (%)': round(pct_global, 2),
            'Taux Citation Urbain (%)': round(pct_urbain, 2),
            'Taux Citation Périurbain (%)': round(pct_periurbain, 2),
            'Statistique Chi2': round(chi2, 3),
            'ddl': dof,
            'p-value': round(p_val, 4),
            'Significatif (p < 0.05)': 'Oui' if p_val < 0.05 else 'Non'
        })
        
    return pd.DataFrame(resultats_multiples)

# ==============================================================================
# 3. EXPORTATION DES RESULTATS DANS UN FICHIER EXCEL
# ==============================================================================
def exporter_livrable_services(df, cols_binaires=None, col_multiple=None, filename='Tableau_6_1_3_Services_Agricoles.xlsx'):
    df_binaires = analyser_services_binaires(df, cols_binaires=cols_binaires)
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df_binaires.to_excel(writer, sheet_name='Services_Accès_Accompagnement', index=False)
        
        if col_multiple and col_multiple in df.columns:
            df_multiples = analyser_reponses_multiples(df, col_multiple=col_multiple)
            df_multiples.to_excel(writer, sheet_name='Prestataires_Reponses_Multiples', index=False)
            
    print(f"✅ Fichier Excel exporté avec succès sous : {filename}")

# ==============================================================================
# 4. EXÉCUTION DU SCRIPT
# ==============================================================================
# Normalisation du nom de la zone si nécessaire
if 'zone' not in df.columns and 'zone_etude' in df.columns:
    df['zone'] = df['zone_etude']

# 1. Vos colonnes binaires
mes_cols_binaires = [df.columns[495], df.columns[43], df.columns[45]]

# 2. Ligne manquante (Mettre None si vous n'avez pas de question à choix multiples)
mes_reponses_multiples = None 

# Affichage des résultats
print("=== 1. SERVICES ET ACCÈS (VARIABLES BINAIRES) ===")
res_binaires = analyser_services_binaires(df, cols_binaires=mes_cols_binaires)
display(res_binaires)

# Exportation Excel
exporter_livrable_services(df, cols_binaires=mes_cols_binaires, col_multiple=mes_reponses_multiples)   
# 1. Vérification des effectifs bruts par zone
print("--- Effectifs bruts dans la colonne zone ---")
print(df['zone'].value_counts(dropna=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# FONCTION : Génération du graphique des services agricoles
# ==============================================================================
def generer_graphique_services(res_binaires, output_img='Fig_6_1_3_Services_Agricoles.png'):
    # Restructuration des données pour Seaborn
    df_plot = res_binaires.melt(
        id_vars=['Indicateur / Service'], 
        value_vars=['Taux Urbain (%)', 'Taux Périurbain (%)'],
        var_name='Zone', 
        value_name='Taux (%)'
    )
    df_plot['Zone'] = df_plot['Zone'].str.replace('Taux ', '').str.replace(' (%)', '')

    sns.set_theme(style='whitegrid', font='DejaVu Sans')
    plt.figure(figsize=(10, 5))
    
    ax = sns.barplot(
        data=df_plot, 
        x='Indicateur / Service', 
        y='Taux (%)', 
        hue='Zone', 
        palette=['#1f77b4', '#d62728']
    )

    plt.title("Taux d'accès aux services agricoles et à l'accompagnement par zone", fontsize=12, fontweight='bold', pad=15)
    plt.ylabel("Proportion (%)", fontsize=11)
    plt.xlabel("", fontsize=11)
    plt.ylim(0, max(df_plot['Taux (%)'].max() + 15, 50))

    # Affichage des pourcentages au-dessus des barres
    for p in ax.patches:
        height = p.get_height()
        if height >= 0:
            ax.annotate(f'{height:.1f}%',
                        (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom',
                        fontsize=9, fontweight='bold',
                        xytext=(0, 3), textcoords='offset points')

    # Raccourcir le texte de l'axe X pour éviter les chevauchements
    labels = [text.get_text()[:30] + '...' if len(text.get_text()) > 30 else text.get_text() for text in ax.get_xticklabels()]
    ax.set_xticklabels(labels, rotation=15, ha='right')

    plt.legend(title='Zone', frameon=True)
    plt.tight_layout()
    plt.savefig(output_img, dpi=300, bbox_inches='tight')
    plt.show()

# ==============================================================================
# EXÉCUTION DE L'AFFICHAGE ET DU GRAPHIQUE
# ==============================================================================
print("=== 1. SERVICES ET ACCÈS (VARIABLES BINAIRES) ===")
res_binaires = analyser_services_binaires(df, cols_binaires=mes_cols_binaires)
display(res_binaires)

# Génération de la figure
generer_graphique_services(res_binaires)

# Exportation Excel
exporter_livrable_services(df, cols_binaires=mes_cols_binaires, col_multiple=mes_reponses_multiples)

## Section 6.2 : Validation et Analyse de l'Indice d'Adoption Agroécologique (IAA)

### 6.2.1 Consistance interne et validation psychométrique de l'indice

#### Objectif académique : Démontrer la cohérence statistique et la fiabilité de l'échelle additive d'adoption agroécologique construite sur 25 items.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

# ==============================================================================
# 1. CONVERSION EXACTE DES NIVEAUX DE PRATIQUE (0, 1, 2, 3)
# ==============================================================================
def nettoyer_et_convertir_niveaux(df_sub):
    """Extrait le chiffre du Niveau (0, 1, 2 ou 3) présent dans le texte."""
    df_clean = df_sub.copy()
    for col in df_clean.columns:
        # Extraction du premier chiffre trouvé dans la chaîne (ex: 'Niveau 2' -> 2)
        extracted = df_clean[col].astype(str).str.extract(r'(\d+)')[0]
        df_clean[col] = pd.to_numeric(extracted, errors='coerce')
    return df_clean

# ==============================================================================
# 2. CALCUL PSYCHOMÉTRIQUE
# ==============================================================================
def calculer_alpha_cronbach(df_items):
    k = df_items.shape[1]
    if k <= 1:
        return np.nan
    item_variances = df_items.var(ddof=1).sum()
    total_variance = df_items.sum(axis=1).var(ddof=1)
    if total_variance == 0 or np.isnan(total_variance):
        return 0.0
    return (k / (k - 1)) * (1 - (item_variances / total_variance))

def analyser_consistance_interne(df_items):
    df_selected = nettoyer_et_convertir_niveaux(df_items)
    cols_items = list(df_selected.columns)
    
    alpha_global = calculer_alpha_cronbach(df_selected)
    score_total = df_selected.sum(axis=1)
    
    resultats_items = []
    for col in cols_items:
        score_hors_item = score_total - df_selected[col]
        corr_item = df_selected[col].corr(score_hors_item)
        
        df_sans_item = df_selected.drop(columns=[col])
        alpha_si_supprime = calculer_alpha_cronbach(df_sans_item)
        impact_alpha = alpha_si_supprime - alpha_global
        
        if np.isnan(corr_item) or corr_item < 0.30:
            diagnostic = "À revoir (r < 0.30)"
        elif impact_alpha > 0.01:
            diagnostic = "À exclure (Augmente α)"
        else:
            diagnostic = "Conserver"
            
        nom_court = str(col).split('?')[0]
        nom_affichable = nom_court[:35] + '...' if len(nom_court) > 35 else nom_court
        
        resultats_items.append({
            'Code Item / Pratique': nom_affichable,
            'Moyenne': round(df_selected[col].mean(), 2),
            'Écart-Type': round(df_selected[col].std(ddof=1), 2),
            'Corrélation Item-Total': round(corr_item, 3) if not np.isnan(corr_item) else np.nan,
            'Alpha si Supprimé': round(alpha_si_supprime, 3),
            'Impact sur Alpha (Δα)': round(impact_alpha, 3),
            'Diagnostic': diagnostic
        })
        
    df_analyse_items = pd.DataFrame(resultats_items)
    
    qualification = "Excellente" if alpha_global >= 0.90 else "Bonne" if alpha_global >= 0.80 else "Acceptable" if alpha_global >= 0.70 else "Insuffisante"
        
    df_synthese = pd.DataFrame([{
        'Indicateur Psychométrique': "Alpha de Cronbach Global (α)",
        'Nombre d\'Items Validés (k)': len(cols_items),
        'Nombre d\'Observations': len(df_selected),
        'Valeur Obtenue': round(alpha_global, 3),
        'Seuil Acceptabilité': "≥ 0.70",
        'Statut Fiabilité': qualification
    }])
    
    return df_synthese, df_analyse_items

# ==============================================================================
# 3. EXPORT ET VISUALISATION
# ==============================================================================
def exporter_et_afficher_cronbach(df_items, filename='Tableau_6_2_1_Alpha_Cronbach_Items.xlsx'):
    df_synthese, df_items_detail = analyser_consistance_interne(df_items)
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df_synthese.to_excel(writer, sheet_name='Synthese_Global_Alpha', index=False)
        df_items_detail.to_excel(writer, sheet_name='Impact_Suppression_Items', index=False)
    
    print("=== SYNTHÈSE GLOBALE DE FIABILITÉ ===")
    display(df_synthese)
    print("\n=== ANALYSE DÉTAILLÉE PAR ITEM (25 PRATIQUES) ===")
    display(df_items_detail)

    colors = ['#d9534f' if (np.isnan(r) or r < 0.30) else '#2ca02c' for r in df_items_detail['Corrélation Item-Total']]

    plt.figure(figsize=(14, 6))
    sns.set_theme(style='whitegrid')
    sns.barplot(
        data=df_items_detail, 
        x='Code Item / Pratique', 
        y='Corrélation Item-Total', 
        palette=colors, 
        hue='Code Item / Pratique', 
        legend=False
    )
    plt.axhline(0.30, color='red', linestyle='--', linewidth=1.5, label='Seuil critique (r = 0.30)')
    plt.title("Corrélation Item-Total des 25 Pratiques Agroécologiques (Échelle 0-3)", fontsize=12, fontweight='bold', pad=15)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.ylabel("Corrélation (r)", fontsize=11)
    plt.tight_layout()
    # À insérer à la fin du code de ton graphique
    plt.savefig('Correlation_Item_Total.png', dpi=300, bbox_inches='tight')
    plt.show()

# ==============================================================================
# EXÉCUTION
# ==============================================================================
indices_25_items = [
    49, 70, 87, 106, 121, 134, 149, 166, 182, 197, 
    213, 230, 245, 260, 276, 292, 307, 323, 338, 354, 
    369, 384, 400, 416, 429
]

df_25_items = df.iloc[:, indices_25_items]
exporter_et_afficher_cronbach(df_25_items)      

In [ ]:
indices_25_items = [
    49, 70, 87, 106, 121, 134, 149, 166, 182, 197, 
    213, 230, 245, 260, 276, 292, 307, 323, 338, 354, 
    369, 384, 400, 416, 429
]

print("=== MODALITÉS BRUTES POUR LES 25 ITEMS ===\n")

for i, idx in enumerate(indices_25_items, 1):
    nom_col = df.columns[idx]
    nom_court = str(nom_col).split('?')[0][:60]
    
    print(f"--- Item {i} (Index {idx}) : {nom_court} ---")
    print(df.iloc[:, idx].value_counts(dropna=False))
    print("-" * 65 + "\n")

### 6.2.2 Distribution globale et variabilité spatiale du score IAA

#### Objectif académique : Analyser le niveau d'adoption globale et tester les disparités d'adoption le long du gradient d'urbanisation.

In [ ]:
# 1. Vérification des effectifs bruts par zone
print("--- Effectifs bruts dans la colonne zone ---")
print(df['zone'].value_counts(dropna=False))

# 2. Vérification des valeurs manquantes dans le score IAA par zone
print("\n--- Nombre de valeurs manquantes (NaN) dans score_agroeco_pct par zone ---")
print(df.groupby('zone')['score_agroeco_pct'].apply(lambda x: x.isna().sum()))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, skew, kurtosis
from matplotlib.gridspec import GridSpec

# ==============================================================================
# 1. IDENTIFICATION DES 25 PRATIQUES ET DÉTECTION DE LA ZONE
# ==============================================================================
# Liste des index exacts des 25 items dans le DataFrame
indices_25_items = [
    49, 70, 87, 106, 121, 134, 149, 166, 182, 197, 
    213, 230, 245, 260, 276, 292, 307, 323, 338, 354, 
    369, 384, 400, 416, 429
]
cols_25_pratiques = df.columns[indices_25_items].tolist()

# Détection de la colonne identifiant ménage (ex: 'Nouveau_id') avec codes U-xx / P-xx
col_id = None
for col in df.columns:
    matches = df[col].astype(str).str.strip().str.upper().str.startswith(('U-', 'P-'))
    if matches.sum() > 50:
        col_id = col
        break

# Correction directe de la variable 'zone' (89 Urbain / 89 Périurbain)
df['zone'] = df[col_id].astype(str).str.strip().str.upper().apply(
    lambda x: 'Urbain' if x.startswith('U') else ('Périurbain' if x.startswith('P') else np.nan)
)

print(f"Colonne d'identifiant utilisée : '{col_id}'")
print("Répartition corrigée de la zone :")
print(df['zone'].value_counts())

# ==============================================================================
# 2. NETTOYAGE ET CALCUL DE L'IAA EN % (SUR 75 POINTS MAX)
# ==============================================================================
df_clean = df.copy()
cols_numeriques = []

for col in cols_25_pratiques:
    # Extraction du premier chiffre (0, 1, 2 ou 3) de la réponse texte
    chiffre_extrait = df_clean[col].astype(str).str.extract(r'(\d)')[0]
    col_num = str(col) + '_num'
    df_clean[col_num] = pd.to_numeric(chiffre_extrait, errors='coerce').fillna(0)
    cols_numeriques.append(col_num)

# Score total (Max = 25 items * 3 = 75 points)
score_max = len(cols_25_pratiques) * 3
df_clean['score_total_points'] = df_clean[cols_numeriques].sum(axis=1)
df_clean['score_agroeco_pct'] = (df_clean['score_total_points'] / score_max) * 100

print(f"\nNettoyage réussi ! Score calculé sur un maximum de {score_max} points.")

# ==============================================================================
# 3. STATISTIQUES DESCRIPTIVES DU SCORE IAA (%)
# ==============================================================================
def extraire_indicateurs(series):
    s_clean = series.dropna()
    q25 = s_clean.quantile(0.25)
    q75 = s_clean.quantile(0.75)
    return pd.Series({
        'Effectif (n)': len(s_clean),
        'Moyenne (%)': round(s_clean.mean(), 2),
        'Écart-Type (%)': round(s_clean.std(ddof=1), 2),
        'Médiane (%)': round(s_clean.median(), 2),
        'Q1_25% (%)': round(q25, 2),
        'Q3_75% (%)': round(q75, 2),
        'IQR (%)': round(q75 - q25, 2),
        'Min (%)': round(s_clean.min(), 2),
        'Max (%)': round(s_clean.max(), 2),
        'Asymétrie (Skewness)': round(skew(s_clean), 3),
        'Aplatissement (Kurtosis)': round(kurtosis(s_clean), 3)
    })

stats_global = extraire_indicateurs(df_clean['score_agroeco_pct']).rename(f'Ensemble (n={len(df_clean)})')
stats_zone = df_clean.groupby('zone')['score_agroeco_pct'].apply(extraire_indicateurs).unstack().T
df_stats = pd.concat([stats_global, stats_zone], axis=1)

# ==============================================================================
# 4. TEST DE MANN-WHITNEY (URBAIN VS PÉRIURBAIN)
# ==============================================================================
score_urbain = df_clean[df_clean['zone'] == 'Urbain']['score_agroeco_pct'].dropna()
score_periurbain = df_clean[df_clean['zone'] == 'Périurbain']['score_agroeco_pct'].dropna()

stat_u, p_value = mannwhitneyu(score_urbain, score_periurbain, alternative='two-sided')
is_sig = p_value < 0.05
interp = "Différence significative du niveau d'adoption selon la zone" if is_sig else "Absence de différence significative d'adoption"

df_mw = pd.DataFrame([{
    'Comparaison': 'Urbain vs. Périurbain',
    'Effectif Urbain (n1)': len(score_urbain),
    'Effectif Périurbain (n2)': len(score_periurbain),
    'Statistique U de Mann-Whitney': round(stat_u, 3),
    'p-value': round(p_value, 4),
    'Significatif (p < 0.05)': 'Oui' if is_sig else 'Non',
    'Interprétation': interp
}])

# ==============================================================================
# 5. EXPORTATION DES TABLEAUX DANS EXCEL
# ==============================================================================
with pd.ExcelWriter('Tableau_6_2_2_Statistiques_IAA.xlsx', engine='openpyxl') as writer:
    df_stats.to_excel(writer, sheet_name='Stats_Descriptives_IAA')
    df_mw.to_excel(writer, sheet_name='Test_Mann_Whitney_Zone', index=False)

print("\n=== STATISTIQUES DESCRIPTIVES DU SCORE IAA (%) ===")
display(df_stats)
print("\n=== TEST DE MANN-WHITNEY ===")
display(df_mw)

# ==============================================================================
# 6. GÉNÉRATION DE LA FIGURE 6.2 (HISTOGRAMME/KDE + BOXPLOTS)
# ==============================================================================
sns.set_theme(style='whitegrid', font='DejaVu Sans')
fig = plt.figure(figsize=(12, 8))
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.18)

ax_hist = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1], sharex=ax_hist)

palette = {'Périurbain': '#1f77b4', 'Urbain': '#2ca02c'}

sns.histplot(
    data=df_clean, x='score_agroeco_pct', hue='zone', kde=True,
    element='step', stat='density', common_norm=False,
    palette=palette, ax=ax_hist, alpha=0.3, linewidth=1.5
)

ax_hist.set_title("A. Distribution de la densité de l'Indice d'Adoption Agroécologique (IAA %)", fontsize=12, fontweight='bold', pad=10)
ax_hist.set_ylabel("Densité de probabilité", fontsize=11)
ax_hist.set_xlabel("")
plt.setp(ax_hist.get_xticklabels(), visible=False)

sns.boxplot(
    data=df_clean, x='score_agroeco_pct', y='zone', palette=palette, ax=ax_box,
    width=0.5, boxprops=dict(alpha=0.7), showmeans=True,
    meanprops={"marker": "o", "markerfacecolor": "red", "markeredgecolor": "red", "markersize": "6"}
)

ax_box.set_title("B. Boxplots comparatifs par zone (Point rouge = Moyenne, Trait central = Médiane)", fontsize=11, fontweight='bold', pad=8)
ax_box.set_xlabel("Indice d'Adoption Agroécologique (%)", fontsize=11)
ax_box.set_ylabel("Zone", fontsize=11)
ax_box.set_xlim(0, 100)

plt.savefig('Fig_6_2_Distribution_IAA_Zone.png', dpi=300, bbox_inches='tight')
plt.show() 

### 6.2.3 Hiérarchisation et cartographie des 25 pratiques agroécologiques

#### Objectif académique : Identifier les leviers d'adoption (Top 5) et les goulots d'étranglement (Flop 5) au sein du système de production.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# 1. CARTOGRAPHIE EXACTE DES 25 ITEMS BRUTS (INDEX 1-À-1 KOBO)
# ==============================================================================
ITEMS_MAPPING = [
    {"idx": 49,  "code": "P01", "nom": "Diversification des cultures", "dimension": "Diversification & Agroforesterie"},
    {"idx": 70,  "code": "P02", "nom": "Rotation des cultures", "dimension": "Diversification & Agroforesterie"},
    {"idx": 87,  "code": "P03", "nom": "Association des cultures", "dimension": "Diversification & Agroforesterie"},
    {"idx": 106, "code": "P04", "nom": "Apport de fumier / compost", "dimension": "Gestion des Sols & Salinité"},
    {"idx": 121, "code": "P05", "nom": "Engrais verts / Cultures de couverture", "dimension": "Gestion des Sols & Salinité"},
    {"idx": 134, "code": "P06", "nom": "Biofertilisants liquides / Inoculants", "dimension": "Gestion des Sols & Salinité"},
    {"idx": 149, "code": "P07", "nom": "Biocontrôle / Lutte intégrée", "dimension": "Protection & Ressources"},
    {"idx": 166, "code": "P08", "nom": "Réduction des intrants chimiques", "dimension": "Protection & Ressources"},
    {"idx": 182, "code": "P09", "nom": "Semences locales sélectionnées / améliorées", "dimension": "Diversification & Agroforesterie"},
    {"idx": 197, "code": "P10", "nom": "Régénération Naturelle Assistée (RNA)", "dimension": "Diversification & Agroforesterie"},
    {"idx": 213, "code": "P11", "nom": "Haies brise-vent & Agroforesterie", "dimension": "Diversification & Agroforesterie"},
    {"idx": 230, "code": "P12", "nom": "Plantes répulsives / compagnes", "dimension": "Protection & Ressources"},
    {"idx": 245, "code": "P13", "nom": "Paillage / Mulching", "dimension": "Gestion des Sols & Salinité"},
    {"idx": 260, "code": "P14", "nom": "Aménagements anti-érosion / salinité", "dimension": "Gestion des Sols & Salinité"},
    {"idx": 276, "code": "P15", "nom": "Recyclage des résidus de culture", "dimension": "Gestion des Sols & Salinité"},
    {"idx": 292, "code": "P16", "nom": "Semis direct / Travail réduit du sol", "dimension": "Gestion des Sols & Salinité"},
    {"idx": 307, "code": "P17", "nom": "Irrigation efficiente", "dimension": "Protection & Ressources"},
    {"idx": 323, "code": "P18", "nom": "Pompage solaire / Énergies renouvelables", "dimension": "Protection & Ressources"},
    {"idx": 338, "code": "P19", "nom": "Intégration agriculture-élevage", "dimension": "Diversification & Agroforesterie"},
    {"idx": 354, "code": "P20", "nom": "Stockage hermétique (Sacs PICS / GrainPro)", "dimension": "Socio-Économie & Gouvernance"},
    {"idx": 369, "code": "P21", "nom": "Suivi visuel des cultures / ravageurs", "dimension": "Protection & Ressources"},
    {"idx": 384, "code": "P22", "nom": "Lutte contre la salinité des sols (tanns)", "dimension": "Gestion des Sols & Salinité"},
    {"idx": 400, "code": "P23", "nom": "Vente en circuits collectifs / durables", "dimension": "Socio-Économie & Gouvernance"},
    {"idx": 416, "code": "P24", "nom": "Partage & co-création de connaissances", "dimension": "Socio-Économie & Gouvernance"},
    {"idx": 429, "code": "P25", "nom": "Gouvernance locale & gestion collective", "dimension": "Socio-Économie & Gouvernance"}
]

# ==============================================================================
# 2. FONCTION D'ANALYSE ET DE CALCUL
# ==============================================================================
def analyser_hierarchie_exacte(df_households, mapping=ITEMS_MAPPING):
    résultats = []
    
    for item in mapping:
        col_name = df_households.columns[item["idx"]]
        
        # Extraction du score numérique (0, 1, 2 ou 3) depuis la modalité texte
        scores_extracted = df_households[col_name].astype(str).str.extract(r'(\d)')[0]
        scores_num = pd.to_numeric(scores_extracted, errors='coerce').fillna(0)
        
        score_moyen = scores_num.mean()
        ecart_type = scores_num.std(ddof=1)
        taux_penetration = (scores_num > 0).mean() * 100.0
        intensite_adoption = (score_moyen / 3.0) * 100.0
        
        résultats.append({
            'Code': item["code"],
            'Pratique Agroécologique': item["nom"],
            'Dimension Agroécologique': item["dimension"],
            'Score Moyen (0-3)': round(score_moyen, 2),
            'Écart-Type': round(ecart_type, 2),
            'Taux de Pénétration (%)': round(taux_penetration, 2),
            'Intensité d\'Adoption (%)': round(intensite_adoption, 2)
        })
        
    df_ranking = pd.DataFrame(résultats).sort_values(by='Intensité d\'Adoption (%)', ascending=False).reset_index(drop=True)
    df_ranking['Rang'] = range(1, len(df_ranking) + 1)
    
    df_ranking['Statut Rôle'] = 'Intermédiaire'
    df_ranking.loc[df_ranking['Rang'] <= 5, 'Statut Rôle'] = "Top 5 (Levier d'adoption)"
    df_ranking.loc[df_ranking['Rang'] > 20, 'Statut Rôle'] = "Flop 5 (Goulot d'étranglement)"
    
    return df_ranking

# ==============================================================================
# 3. EXPORTATION DU LIVRABLE EXCEL (3 ONGLETS)
# ==============================================================================
def exporter_livrable_excel(df_ranking, filename='Tableau_6_2_3_Top_Flop_Pratiques_Reelles.xlsx'):
    top_5 = df_ranking[df_ranking['Rang'] <= 5]
    flop_5 = df_ranking[df_ranking['Rang'] > 20]
    
    df_top_flop_synth = pd.concat([top_5, flop_5], axis=0)
    
    df_dim_summary = df_ranking.groupby('Dimension Agroécologique').agg(
        Nombre_Pratiques=('Code', 'count'),
        Moyenne_Intensité_Adoption=('Intensité d\'Adoption (%)', 'mean'),
        Moyenne_Taux_Pénétration=('Taux de Pénétration (%)', 'mean')
    ).round(2).reset_index().sort_values(by='Moyenne_Intensité_Adoption', ascending=False)

    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df_ranking.to_excel(writer, sheet_name='Classement_25_Pratiques', index=False)
        df_top_flop_synth.to_excel(writer, sheet_name='Top5_Leviers_Flop5_Goulots', index=False)
        df_dim_summary.to_excel(writer, sheet_name='Synthese_4_Dimensions', index=False)
        
    print(f"--> Export Excel réussi : '{filename}'")

# ==============================================================================
# 4. GÉNÉRATION DE LA FIGURE 6.3 (RANK PLOT)
# ==============================================================================
def generer_figure_6_3(df_ranking, output_img='Fig_6_3_Rank_Plot_25_Pratiques.png'):
    plt.figure(figsize=(12, 10))
    sns.set_theme(style='whitegrid', font='DejaVu Sans')

    palette = {
        'Diversification & Agroforesterie': '#276419',
        'Gestion des Sols & Salinité': '#8c510a',
        'Protection & Ressources': '#01665e',
        'Socio-Économie & Gouvernance': '#7fbc41'
    }

    ax = sns.barplot(
        data=df_ranking,
        y='Pratique Agroécologique',
        x='Intensité d\'Adoption (%)',
        hue='Dimension Agroécologique',
        dodge=False,
        palette=palette,
        edgecolor='black',
        linewidth=0.5
    )

    for p in ax.patches:
        width = p.get_width()
        if width > 0:
            ax.annotate(
                f"{width:.1f}%",
                (width + 1.0, p.get_y() + p.get_height() / 2.0),
                ha='left',
                va='center',
                fontsize=9,
                fontweight='bold',
                color='#222222'
            )

    plt.title("Fig 6.3 : Hiérarchisation des 25 pratiques agroécologiques selon l'intensité d'adoption (%)", fontsize=12, fontweight='bold', pad=15)
    plt.xlabel("Intensité d'Adoption Agroécologique (%)", fontsize=11, fontweight='bold')
    plt.ylabel("Pratiques Agroécologiques (Classées du Rang 1 au Rang 25)", fontsize=11, fontweight='bold')
    plt.xlim(0, 100)

    # Lignes de démarcation Top 5 / Flop 5
    plt.axhline(4.5, color='green', linestyle='--', linewidth=1, alpha=0.7)
    plt.axhline(19.5, color='red', linestyle='--', linewidth=1, alpha=0.7)

    plt.text(78, 2, "▲ TOP 5 LEVIERS", fontsize=9, fontweight='bold', color='green', bbox=dict(boxstyle="round,pad=0.3", fc="#e6ffe6", ec="green", lw=1))
    plt.text(76, 22, "▼ FLOP 5 GOULOTS", fontsize=9, fontweight='bold', color='red', bbox=dict(boxstyle="round,pad=0.3", fc="#ffcccc", ec="red", lw=1))

    plt.legend(title="Dimension Agroécologique", loc='lower right', frameon=True)
    plt.tight_layout()
    plt.savefig(output_img, dpi=300, bbox_inches='tight')
    plt.show()

# ==============================================================================
# 5. EXECUTION COMPLÈTE
# ==============================================================================
df_ranking_exact = analyser_hierarchie_exacte(df)

print("=== CLASSEMENT EXACT DES 25 PRATIQUES AGROÉCOLOGIQUES ===")
display(df_ranking_exact)

exporter_livrable_excel(df_ranking_exact)
generer_figure_6_3(df_ranking_exact)

## Section 6.3 : Typologie Structurelle et Fonctionnelle des Exploitations

### 6.3.1 Démarche factorielle et classification hiérarchique

#### Objectif académique : Réduire la dimensionnalité des données d'exploitation et identifier la structure sous-jacente des profil d'adoptants.

In [ ]:
# Recherche ciblée : affiche uniquement l'index et le nom des colonnes clés
mots_cles = ['superficie', 'revenu', 'actif', 'maraîch', 'autoconsommation', 'score']

for i, col in enumerate(df.columns):
    if any(m in col.lower() for m in mots_cles):
        print(f"Index {i} : {col}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# ==============================================================================
# 1. SÉLECTION ET NETTOYAGE DES VARIABLES ACTIVES PAR INDEX
# ==============================================================================
# Index identifiés : 481 (Superficie), 445 (Score agroéco), 464 (Autoconsommation), 491 (Revenu net), 24 (Maraîchage)
indices_variables = [481, 445, 464, 491, 24]
VARIABLES_ACTIVES = [df.columns[i] for i in indices_variables]

df_pca = df[VARIABLES_ACTIVES].copy()

# Nettoyage et conversion numérique
for col in df_pca.columns:
    if df_pca[col].dtype == 'object':
        df_pca[col] = df_pca[col].astype(str).str.contains('1|Oui|True|Maraîchage', case=False, regex=True).astype(int)
    else:
        df_pca[col] = pd.to_numeric(df_pca[col], errors='coerce')
        df_pca[col] = df_pca[col].replace([99, 999, -99], np.nan)

# Imputation par la médiane pour combler les valeurs manquantes
df_pca = df_pca.fillna(df_pca.median())

# Standardisation Z-Score (moyenne = 0, std = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca)
df_scaled = pd.DataFrame(X_scaled, columns=VARIABLES_ACTIVES, index=df.index)

# ==============================================================================
# 2. ANALYSE EN COMPOSANTES PRINCIPALES (ACP)
# ==============================================================================
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

valeurs_propres = pca.explained_variance_
var_expliquee_pct = pca.explained_variance_ratio_ * 100.0
var_cumulee_pct = np.cumsum(var_expliquee_pct)

# Critère de Kaiser (λ ≥ 1)
axes_retenus = ["Retenu (Kaiser)" if ev >= 1.0 else "Rejeté" for ev in valeurs_propres]

df_variance_acp = pd.DataFrame({
    'Axe Factoriel': [f"Axe {i+1} (F{i+1})" for i in range(len(valeurs_propres))],
    'Valeur Propre (λ)': np.round(valeurs_propres, 3),
    'Variance Expliquée (%)': np.round(var_expliquee_pct, 2),
    'Variance Cumulée (%)': np.round(var_cumulee_pct, 2),
    'Décision (Critère Kaiser λ ≥ 1)': axes_retenus
})

# Nombre d'axes retenus (au moins 2)
n_axes_kaiser = int(np.sum(valeurs_propres >= 1.0))
n_axes_retenus = max(n_axes_kaiser, 2)
X_coordonnees = X_pca[:, :n_axes_retenus]

# Matrice des corrélations (Loadings)
loadings = pca.components_[:n_axes_retenus, :].T * np.sqrt(valeurs_propres[:n_axes_retenus])
labels_lisibles = ['Superficie (ha)', 'Score Agroéco (%)', 'Autoconsommation (%)', 'Revenu Agri Net (FCFA)', 'Pratique Maraîchage']
df_loadings = pd.DataFrame(
    np.round(loadings, 3),
    index=labels_lisibles,
    columns=[f"Axe F{i+1}" for i in range(n_axes_retenus)]
)

# ==============================================================================
# 3. CLASSIFICATION ASCENDANTE HIÉRARCHIQUE (CAH)
# ==============================================================================
k_clusters = 3
Z_linkage = linkage(X_coordonnees, method='ward', metric='euclidean')
cluster_labels = fcluster(Z_linkage, t=k_clusters, criterion='maxclust')

df_resultats = df.copy()
df_resultats['Cluster_Typologie'] = [f"Profil {c}" for c in cluster_labels]

# ==============================================================================
# 4. EXPORTATION DU LIVRABLE EXCEL (3 ONGLETS)
# ==============================================================================
excel_filename = 'Tableau_6_3_1_Variance_Explication_ACP.xlsx'
with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df_variance_acp.to_excel(writer, sheet_name='Variance_Explication_ACP', index=False)
    df_loadings.to_excel(writer, sheet_name='Correlations_Variables_Axes')
    df_resultats.to_excel(writer, sheet_name='Affectation_Menages_Clusters', index_label='ID_Menage')

print(f"--> Livrable Excel sauvegardé : '{excel_filename}'")

# ==============================================================================
# 5. GÉNÉRATION DU DENDROGRAMME (FIGURE 6.4)
# ==============================================================================
plt.close('all')
fig, ax = plt.subplots(figsize=(12, 7))
sns.set_theme(style='white', font='DejaVu Sans')

hauteur_coupe = (Z_linkage[-k_clusters, 2] + Z_linkage[-k_clusters + 1, 2]) / 2.0

dendrogram(
    Z_linkage,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90.,
    leaf_font_size=10.,
    show_contracted=True,
    color_threshold=hauteur_coupe,
    above_threshold_color='#555555',
    ax=ax
)

ax.axhline(
    y=hauteur_coupe,
    color='red',
    linestyle='--',
    linewidth=1.5,
    label=f'Ligne de coupe de Ward (K = {k_clusters} profils, h = {hauteur_coupe:.2f})'
)

plt.title("Fig 6.4 : Dendrogramme de la Classification Ascendante Hiérarchique (Méthode de Ward)", fontsize=12, fontweight='bold', pad=15)
plt.xlabel(f"Agrégation des ménages / Clusters d'exploitations (N={len(df)})", fontsize=11, fontweight='bold')
plt.ylabel("Distance Euclidienne d'Agrégation (Inertie Inter-classe)", fontsize=11, fontweight='bold')

leg = plt.legend(loc='upper right', frameon=True, facecolor='white', framealpha=1.0, fontsize=10)
leg.get_frame().set_edgecolor('#cccccc')

plt.tight_layout()

img_filename = 'Fig_6_4_Dendrogramme_CAH.png'
plt.savefig(img_filename, dpi=300, bbox_inches='tight')
plt.show()

print(f"--> Figure sauvegardée : '{img_filename}'")

# Affichage des résultats dans le notebook
print("\n=== VARIANCE EXPLIQUÉE ACP ===")
display(df_variance_acp)

print("\n=== MATRICE DES CORRÉLATIONS (LOADINGS) ===")
display(df_loadings)

### 6.3.2 Caractérisation et profilage des classes (Clusters)

#### Objectif académique : Décrire chaque type d'exploitation selon ses attributs dominants (valeurs-tests).

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# ==============================================================================
# 1. SÉLECTION, NETTOYAGE ET PRÉPARATION DES DONNÉES
# ==============================================================================
# Index des variables actives : 481 (Superficie), 445 (Score Agro), 464 (Autoconsommation), 491 (Revenu), 24 (Maraîchage)
idx_continues = [481, 445, 464, 491]
idx_binaires  = [24]

cont_vars = [df.columns[i] for i in idx_continues]
bin_vars  = [df.columns[i] for i in idx_binaires]
active_cols = cont_vars + bin_vars

# Nettoyage et conversion numérique
for col in active_cols:
    if col in bin_vars or df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.contains('1|Oui|True|Maraîchage', case=False, regex=True).astype(int)
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([99, 999, -99], np.nan)
    df[col] = df[col].fillna(df[col].median())

# Standardisation Z-Score (Moyenne=0, Std=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[active_cols])

# ==============================================================================
# 2. ANALYSE EN COMPOSANTES PRINCIPALES (ACP)
# ==============================================================================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

valeurs_propres = pca.explained_variance_
var_expliquee_pct = pca.explained_variance_ratio_ * 100.0
var_cumulee_pct = np.cumsum(var_expliquee_pct)

df_variance_acp = pd.DataFrame({
    'Axe Factoriel': ['Axe 1 (F1)', 'Axe 2 (F2)'],
    'Valeur Propre (λ)': np.round(valeurs_propres, 3),
    'Variance Expliquée (%)': np.round(var_expliquee_pct, 2),
    'Variance Cumulée (%)': np.round(var_cumulee_pct, 2),
    'Décision (Kaiser λ ≥ 1)': ["Retenu" if ev >= 1.0 else "Rejeté" for ev in valeurs_propres]
})

# Matrice des corrélations (Loadings)
labels_visuels = ['Superficie (ha)', 'Score Agroéco (%)', 'Autoconsommation (%)', 'Revenu Agri Net', 'Maraîchage']
loadings = pca.components_.T * np.sqrt(valeurs_propres)
df_loadings = pd.DataFrame(np.round(loadings, 3), index=labels_visuels, columns=['Axe F1', 'Axe F2'])

# ==============================================================================
# 3. CLASSIFICATION ASCENDANTE HIÉRARCHIQUE (CAH) & NOMMAGE
# ==============================================================================
k_clusters = 3
Z_linkage = linkage(X_pca, method='ward', metric='euclidean')
cluster_ids = fcluster(Z_linkage, t=k_clusters, criterion='maxclust')

mapping_typologie = {
    1: 'Type 1 : Extensifs Vivriers',
    2: 'Type 2 : Maraîchers Agroécologiques',
    3: 'Type 3 : Profils Autoconsommation'
}
df['Cluster_Label'] = [mapping_typologie.get(c, f"Profil {c}") for c in cluster_ids]

# ==============================================================================
# 4. FONCTIONS DE CALCUL DES VALEURS-TESTS (LEBART / MORINEAU)
# ==============================================================================
def calculer_valeurs_tests_continues(df, group_col, var_cols):
    N = len(df)
    results = []
    for var in var_cols:
        mean_global = df[var].mean()
        std_global = df[var].std(ddof=0)
        for cluster_label, group in df.groupby(group_col):
            nk = len(group)
            mean_k = group[var].mean()
            std_k = group[var].std(ddof=1)
            denom = np.sqrt(((N - nk) / (N - 1)) * ((std_global**2) / nk)) if std_global > 0 and nk < N else 1
            v_test = (mean_k - mean_global) / denom if std_global > 0 else 0.0
            p_val = 2 * (1 - stats.norm.cdf(abs(v_test)))
            results.append({
                'Variable': var, 'Cluster': cluster_label,
                'Moyenne Cluster': round(mean_k, 2), 'Moyenne Globale': round(mean_global, 2),
                'Écart-Type Cluster': round(std_k, 2), 'Valeur-Test (v)': round(v_test, 3),
                'p-value': round(p_val, 4), 'Significatif (|v| ≥ 1.96)': 'Oui' if abs(v_test) >= 1.96 else 'Non',
                'Sens': 'Sur-représenté (+)' if v_test >= 1.96 else ('Sous-représenté (-)' if v_test <= -1.96 else 'Neutre')
            })
    return pd.DataFrame(results)

def calculer_valeurs_tests_binaires(df, group_col, binary_cols):
    N = len(df)
    results = []
    for var in binary_cols:
        p_global = df[var].mean()
        for cluster_label, group in df.groupby(group_col):
            nk = len(group)
            p_k = group[var].mean()
            denom = np.sqrt(((N - nk) / (N - 1)) * ((p_global * (1 - p_global)) / nk)) if 0 < p_global < 1 and nk < N else 1
            v_test = (p_k - p_global) / denom if 0 < p_global < 1 else 0.0
            p_val = 2 * (1 - stats.norm.cdf(abs(v_test)))
            results.append({
                'Variable': var, 'Cluster': cluster_label,
                'Proportion Cluster (%)': round(p_k * 100, 2), 'Proportion Globale (%)': round(p_global * 100, 2),
                'Valeur-Test (v)': round(v_test, 3), 'p-value': round(p_val, 4),
                'Significatif (|v| ≥ 1.96)': 'Oui' if abs(v_test) >= 1.96 else 'Non',
                'Sens': 'Sur-représenté (+)' if v_test >= 1.96 else ('Sous-représenté (-)' if v_test <= -1.96 else 'Neutre')
            })
    return pd.DataFrame(results)

# ==============================================================================
# 5. EXPORTATION DES LIVRABLES EXCEL
# ==============================================================================
with pd.ExcelWriter('Tableau_6_3_1_Variance_Explication_ACP.xlsx', engine='openpyxl') as writer:
    df_variance_acp.to_excel(writer, sheet_name='Variance_Explication_ACP', index=False)
    df_loadings.to_excel(writer, sheet_name='Correlations_Variables_Axes')

df_vtest_cont = calculer_valeurs_tests_continues(df, 'Cluster_Label', cont_vars)
df_vtest_bin = calculer_valeurs_tests_binaires(df, 'Cluster_Label', bin_vars)

with pd.ExcelWriter('Tableau_6_3_2_Profils_Clusters_Valeurs_Tests.xlsx', engine='openpyxl') as writer:
    df.groupby('Cluster_Label')[cont_vars].mean().T.to_excel(writer, sheet_name='Moyennes_Continues')
    (df.groupby('Cluster_Label')[bin_vars].mean().T * 100).to_excel(writer, sheet_name='Proportions_Binaires')
    df_vtest_cont.to_excel(writer, sheet_name='Valeurs_Tests_Continues', index=False)
    df_vtest_bin.to_excel(writer, sheet_name='Valeurs_Tests_Binaires', index=False)

print("--> Livrables Excel sauvegardés avec succès.")

# ==============================================================================
# 6. GÉNÉRATION DE LA FIGURE 6.4 : DENDROGRAMME CAH
# ==============================================================================
plt.close('all')
fig, ax = plt.subplots(figsize=(12, 6.5))
sns.set_theme(style='white', font='DejaVu Sans')

hauteur_coupe = (Z_linkage[-k_clusters, 2] + Z_linkage[-k_clusters + 1, 2]) / 2.0

dendrogram(
    Z_linkage, truncate_mode='lastp', p=30, leaf_rotation=90., leaf_font_size=10.,
    show_contracted=True, color_threshold=hauteur_coupe, above_threshold_color='#555555', ax=ax
)

ax.axhline(y=hauteur_coupe, color='red', linestyle='--', linewidth=1.5,
           label=f'Ligne de coupe de Ward (K = {k_clusters} profils, h = {hauteur_coupe:.2f})')

plt.title("Fig 6.4 : Dendrogramme de la Classification Ascendante Hiérarchique (Méthode de Ward)", fontsize=11, fontweight='bold', pad=15)
plt.xlabel(f"Agrégation des ménages / Clusters d'exploitations (N={len(df)})", fontsize=10, fontweight='bold')
plt.ylabel("Distance Euclidienne d'Agrégation (Inertie Inter-classe)", fontsize=10, fontweight='bold')
plt.legend(loc='upper right', frameon=True, facecolor='white', framealpha=1.0)
plt.tight_layout()
plt.savefig('Fig_6_4_Dendrogramme_CAH.png', dpi=300, bbox_inches='tight')
plt.show()

# ==============================================================================
# 7. GÉNÉRATION DE LA FIGURE 6.5 : PLAN FACTORIEL & ELLIPSES DE CONFIANCE (95 %)
# ==============================================================================
def tracer_ellipse_confiance(x, y, ax, n_std=2.447, facecolor='none', **kwargs):
    if x.size < 3: return ax
    cov = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1]) if (cov[0, 0] * cov[1, 1]) > 0 else 0
    ell_radius_x, ell_radius_y = np.sqrt(1 + pearson), np.sqrt(1 - pearson)
    ellipse = Ellipse((0, 0), width=ell_radius_x * 2, height=ell_radius_y * 2, facecolor=facecolor, **kwargs)
    scale_x, scale_y = np.sqrt(cov[0, 0]) * n_std, np.sqrt(cov[1, 1]) * n_std
    transf = transforms.Affine2D().rotate_deg(45).scale(scale_x, scale_y).translate(np.mean(x), np.mean(y))
    ellipse.set_transform(transf + ax.transData)
    return ax.add_patch(ellipse)

plt.close('all')
plt.figure(figsize=(11, 8.5))
sns.set_theme(style='whitegrid', font='DejaVu Sans')
ax = plt.gca()

df_plot = pd.DataFrame({'F1': X_pca[:, 0], 'F2': X_pca[:, 1], 'Cluster': df['Cluster_Label']})

# Palette dynamique pour éviter le gris
palette_couleurs = ['#2ca02c', '#d62728', '#1f77b4'] # Vert, Rouge, Bleu
clusters_uniques = sorted(df_plot['Cluster'].unique())
dict_couleurs = {cluster: palette_couleurs[i % len(palette_couleurs)] for i, cluster in enumerate(clusters_uniques)}

# Nuage de points, centroïdes et ellipses
for label, group in df_plot.groupby('Cluster'):
    c = dict_couleurs[label]
    ax.scatter(group['F1'], group['F2'], label=label, color=c, alpha=0.7, edgecolors='k', s=55)
    ax.scatter(group['F1'].mean(), group['F2'].mean(), color=c, marker='X', s=220, edgecolors='black', linewidth=1.5, zorder=5)
    tracer_ellipse_confiance(group['F1'].values, group['F2'].values, ax, n_std=2.447, facecolor=c, alpha=0.15, edgecolor=c, linewidth=2, linestyle='--')

# Flèches des variables (Loadings)
echelle_fleches = 2.5
for i, var in enumerate(labels_visuels):
    ax.arrow(0, 0, loadings[i, 0] * echelle_fleches, loadings[i, 1] * echelle_fleches,
             head_width=0.12, head_length=0.18, fc='darkred', ec='darkred', alpha=0.8, length_includes_head=True)
    ax.text(loadings[i, 0] * echelle_fleches * 1.15, loadings[i, 1] * echelle_fleches * 1.15, var,
            color='black', ha='center', va='center', fontsize=9, fontweight='bold')

plt.axhline(0, color='gray', linestyle=':', linewidth=1)
plt.axvline(0, color='gray', linestyle=':', linewidth=1)

plt.title(f"Fig 6.5 : Plan Factoriel (F1 vs F2) et Ellipses de Confiance à 95 % par Cluster (N={len(df)})", fontsize=11, fontweight='bold', pad=15)
plt.xlabel(f"Axe Factoriel 1 - Maraîchage & Agroécologie ({var_expliquee_pct[0]:.2f} % de variance)", fontsize=10, fontweight='bold')
plt.ylabel(f"Axe Factoriel 2 - Dimension Économique & Foncier ({var_expliquee_pct[1]:.2f} % de variance)", fontsize=10, fontweight='bold')
plt.legend(title="Typologies d'exploitations (CAH)", loc='upper left', frameon=True, fontsize=9)
plt.tight_layout()
plt.savefig('Fig_6_5_Plan_Factoriel_Clusters.png', dpi=300, bbox_inches='tight')
plt.show()

print("--> Figures 6.4 et 6.5 sauvegardées avec succès.")

### 6.3.3 Analyse comparative des performances socio-économiques entre types

#### Objectif académique : Évaluer si la typologie issue de la CAH explique de manière significative les écarts de performance économique et de sécurité d'existence.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# ==============================================================================
# 1. Tests d'hypothèses, ANOVA / Kruskal-Wallis et Tailles d'effet
# ==============================================================================
def analyser_performances_clusters(df, group_col, target_vars, alpha=0.05):
    """
    Réalise une analyse comparative inter-clusters pour chaque variable de performance :
    - Tests d'hypothèses (Shapiro-Wilk pour normalité, Levene pour homoscédasticité)
    - Test paramétrique (ANOVA F-test + Eta-carré η²)
    - Test non paramétrique (Kruskal-Wallis H-test + Epsilon-carré ε²)
    - Tests post-hoc (Tukey-HSD et Dunn avec correction de Bonferroni)
    """
    N = len(df)
    results_summary = []
    anova_kw_results = []
    posthoc_tukey_list = []
    posthoc_dunn_list = []

    for var in target_vars:
        # Statistique descriptive par cluster
        for name, group in df.groupby(group_col):
            results_summary.append({
                'Variable': var,
                'Cluster': name,
                'Effectif (n)': len(group),
                'Moyenne': round(group[var].mean(), 2),
                'Écart-type': round(group[var].std(), 2),
                'Médiane': round(group[var].median(), 2),
                'Min': round(group[var].min(), 2),
                'Max': round(group[var].max(), 2),
                'Ecart_Interquartile (IQR)': round(group[var].quantile(0.75) - group[var].quantile(0.25), 2)
            })

        groups_dict = {name: group[var].dropna().values for name, group in df.groupby(group_col)}
        group_names = list(groups_dict.keys())
        groups_list = list(groups_dict.values())
        k = len(group_names)

        # Vérification des conditions d'application
        shapiro_pvals = {name: stats.shapiro(data)[1] if len(data) >= 3 else 0.0 for name, data in groups_dict.items()}
        is_normal = all(p > alpha for p in shapiro_pvals.values())

        levene_stat, levene_p = stats.levene(*groups_list)
        is_homoscedastic = levene_p > alpha

        # Tests globaux
        f_stat, f_pval = stats.f_oneway(*groups_list)
        h_stat, h_pval = stats.kruskal(*groups_list)

        # Taille d'effet Eta-carré (η²) pour l'ANOVA
        overall_mean = df[var].mean()
        ss_between = sum(len(g) * (np.mean(g) - overall_mean)**2 for g in groups_list)
        ss_total = sum(np.sum((g - overall_mean)**2) for g in groups_list)
        eta_squared = ss_between / ss_total if ss_total > 0 else 0.0

        # Taille d'effet Epsilon-carré (ε²) pour Kruskal-Wallis
        epsilon_squared = (h_stat - k + 1) / (N - k) if (N - k) > 0 else 0.0

        anova_kw_results.append({
            'Variable': var,
            'Normalité (Shapiro-Wilk p > 0.05)': 'Oui' if is_normal else 'Non',
            'Homoscédasticité (Levene p-val)': round(levene_p, 4),
            'Homoscédasticité Respectée': 'Oui' if is_homoscedastic else 'Non',
            'ANOVA F-stat': round(f_stat, 3),
            'ANOVA p-value': '< 0.0001' if f_pval < 0.0001 else round(f_pval, 4),
            'ANOVA Eta-carré (η²)': round(eta_squared, 4),
            'Kruskal-Wallis H-stat': round(h_stat, 3),
            'Kruskal p-value': '< 0.0001' if h_pval < 0.0001 else round(h_pval, 4),
            'Kruskal Epsilon-carré (ε²)': round(epsilon_squared, 4),
            'Test Décisionnel Recommandé': 'ANOVA F-Test' if (is_normal and is_homoscedastic) else 'Kruskal-Wallis H-Test'
        })

        # Test Post-Hoc 1 : Tukey HSD (Paramétrique)
        tukey = pairwise_tukeyhsd(endog=df[var], groups=df[group_col], alpha=alpha)
        tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])
        tukey_df.insert(0, 'Variable', var)
        posthoc_tukey_list.append(tukey_df)

        # Test Post-Hoc 2 : Test de Dunn avec correction de Bonferroni (Non Paramétrique)
        df_ranks = df.copy()
        df_ranks['rank'] = stats.rankdata(df_ranks[var])

        pairs = [(group_names[i], group_names[j]) for i in range(k) for j in range(i + 1, k)]
        num_comparisons = len(pairs)

        for g1, g2 in pairs:
            r1 = df_ranks[df_ranks[group_col] == g1]['rank']
            r2 = df_ranks[df_ranks[group_col] == g2]['rank']

            n1, n2 = len(r1), len(r2)
            mean_r1, mean_r2 = r1.mean(), r2.mean()

            se = np.sqrt((N * (N + 1) / 12.0) * (1.0 / n1 + 1.0 / n2))
            z_val = (mean_r1 - mean_r2) / se
            p_raw = 2 * (1 - stats.norm.cdf(abs(z_val)))
            p_bonf = min(1.0, p_raw * num_comparisons)

            posthoc_dunn_list.append({
                'Variable': var,
                'Groupe 1': g1,
                'Groupe 2': g2,
                'Différence Moyenne Rangs': round(mean_r1 - mean_r2, 2),
                'Statistique Z': round(z_val, 3),
                'p-value brute': round(p_raw, 5),
                'p-value ajustée (Bonferroni)': '< 0.0001' if p_bonf < 0.0001 else round(p_bonf, 4),
                'Significatif (α=0.05)': 'Oui' if p_bonf < alpha else 'Non'
            })

    return (
        pd.DataFrame(results_summary),
        pd.DataFrame(anova_kw_results),
        pd.concat(posthoc_tukey_list, ignore_index=True),
        pd.DataFrame(posthoc_dunn_list)
    )

# ==============================================================================
# 2. Exportation des livrables Excel
# ==============================================================================
def exporter_livrable_anova(df, group_col='Cluster_Label', target_vars=None, filename='Tableau_6_3_3_Anova_Performances_Clusters.xlsx'):
    if target_vars is None:
        target_vars = [df.columns[491], df.columns[481], df.columns[464]]  # Revenu, Superficie, Autoconsommation par exemple
        
    df_sum, df_tests, df_tukey, df_dunn = analyser_performances_clusters(df, group_col, target_vars)

    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df_sum.to_excel(writer, sheet_name='Descriptifs_Performances', index=False)
        df_tests.to_excel(writer, sheet_name='Tests_ANOVA_Kruskal', index=False)
        df_tukey.to_excel(writer, sheet_name='PostHoc_Tukey_HSD', index=False)
        df_dunn.to_excel(writer, sheet_name='PostHoc_Dunn_Bonferroni', index=False)

    print(f"--> Fichier Excel généré avec succès : '{filename}'")

# Exécution de la fonction avec vos colonnes d'intérêt
exporter_livrable_anova(df, group_col='Cluster_Label', target_vars=cont_vars)

## Section 6.4 : Distribution Spatiale et Profil de la Diversité Alimentaire (HDDS et WDDS)

### 6.4.1 Score de Diversité Alimentaire des Ménages (HDDS)

#### Objectif académique : Mesurer l'accès économique et physique aux groupes d'aliments au niveau du ménage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==============================================================================
# 1. EXTRACTION DES 12 GROUPES FAO PAR INDEX ET RENOMMAGE PROPRE
# ==============================================================================
# Mapping des index exacts [448 à 459] avec des libellés lisibles pour le graphique
idx_fao = list(range(448, 460))
fao_cols_originales = [df.columns[i] for i in idx_fao]

labels_fao_courts = [
    '1. Céréales', 
    '2. Racines & Tubercules', 
    '3. Légumes', 
    '4. Fruits',
    '5. Viandes & Volailles', 
    '6. Œufs', 
    '7. Poissons & Fruits de mer',
    '8. Légumineuses & Graines', 
    '9. Produits Laitiers', 
    '10. Huiles & Graisses',
    '11. Sucres & Miels', 
    '12. Condiments & Boissons'
]

# Nettoyage binaire des 12 colonnes (1 = Consommé, 0 = Non consommé)
for col in fao_cols_originales:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.contains('1|Oui|True', case=False, regex=True).astype(int)
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Dictionnaire de correspondance pour renommer les colonnes
dict_rename = dict(zip(fao_cols_originales, labels_fao_courts))
df_fao = df[fao_cols_originales].rename(columns=dict_rename)

# Ajout de la zone si présente (ex: colonne 'zone' ou 'Milieu')
col_zone = 'zone' if 'zone' in df.columns else df.columns[0] # Ajuster si besoin
df_fao[col_zone] = df[col_zone]

# ==============================================================================
# 2. CALCUL DES FRÉQUENCES DE CONSOMMATION (%)
# ==============================================================================
freq_zone = df_fao.groupby(col_zone)[labels_fao_courts].mean() * 100
freq_global = df_fao[labels_fao_courts].mean() * 100

df_freq_groupes = freq_zone.T
df_freq_groupes['Global'] = freq_global
df_freq_groupes = df_freq_groupes.reset_index().rename(columns={'index': 'Groupe_Alimentaire'})

print("=== FREQUENCES DE CONSOMMATION DES 12 GROUPES FAO (%) ===")
print(df_freq_groupes.to_string(index=False))

# ==============================================================================
# 3. GENERATION DU SPIDER PLOT (FIG 6.6)
# ==============================================================================
plt.close('all')

categories = df_freq_groupes['Groupe_Alimentaire'].tolist()
N = len(categories)

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

zones_uniques = df_fao[col_zone].unique()
couleurs = ['#1f77b4', '#d62728', '#2ca02c']

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Positionnement des 12 noms d'aliments autour du cercle
plt.xticks(angles[:-1], categories, size=8.5, fontweight='bold', color='#1a252f')

ax.set_rlabel_position(0)
plt.yticks([20, 40, 60, 80, 100], ["20%", "40%", "60%", "80%", "100%"], color="grey", size=8)
plt.ylim(0, 100)

# Traçage des courbes par zone
for i, z in enumerate(zones_uniques):
    vals = df_freq_groupes[z].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, label=f'Zone {z}', color=couleurs[i % len(couleurs)])
    ax.fill(angles, vals, color=couleurs[i % len(couleurs)], alpha=0.15)

plt.title(f"Fig 6.6 : Profil de Consommation Alimentaire des Ménages (HDDS)\nComparaison selon le Milieu (N = {len(df)})",
          size=11, weight='bold', pad=30)
plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1), frameon=True)

plt.tight_layout()
plt.savefig("Fig_6_6_SpiderPlot_HDDS_Zone.png", dpi=300, bbox_inches='tight')
plt.show()

### 6.4.2 Score de Diversité Alimentaire des Femmes (WDDS / MDD-W)

#### Objectif académique : Évaluer la qualité nutritionnelle individuelle des femmes en âge de procréer (15–49 ans) et la prévalence de l'adéquation en micronutriments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# 1. EXTRACTION DES 10 GROUPES FAO WDDS / MDD-W (INDEX 467 À 476)
# ==============================================================================
# Extraction automatique des 10 colonnes par leurs index exacts
idx_wdds = list(range(467, 477))
wdds_cols_originales = [df.columns[i] for i in idx_wdds]

labels_wdds_courts = [
    '1. Céréales, racines & tubercules',
    '2. Légumineuses (niébé, haricots)',
    '3. Noix & graines (arachide, cajou)',
    '4. Produits laitiers',
    '5. Viandes, volailles & poissons',
    '6. Œufs',
    '7. Légumes-feuilles vert foncé',
    '8. Fruits & légumes riches en vit. A',
    '9. Autres légumes',
    '10. Autres fruits'
]

# Nettoyage binaire (1 = Consommé, 0 = Non consommé)
for col in wdds_cols_originales:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.contains('1|Oui|True', case=False, regex=True).astype(int)
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Dictionnaire de renommage
dict_rename_wdds = dict(zip(wdds_cols_originales, labels_wdds_courts))
df_wdds = df[wdds_cols_originales].rename(columns=dict_rename_wdds)

# Détection de la colonne zone/milieu
col_zone = 'zone' if 'zone' in df.columns else df.columns[0]
df_wdds[col_zone] = df[col_zone]

# ==============================================================================
# 2. CALCUL DU SCORE WDDS (0-10) ET SEUIL MDD-W (>= 5 GROUPES)
# ==============================================================================
# Score continu WDDS
df['wdds_score'] = df_wdds[labels_wdds_courts].sum(axis=1)

# Indicateur binaire MDD-W (Minimum Dietary Diversity for Women)
df['mddw_atteint'] = np.where(df['wdds_score'] >= 5, 'MDD-W Atteint (≥5)', 'Non Atteint (<5)')

# ==============================================================================
# 3. STATISTIQUES DESCRIPTIVES ET TABLEAUX D'EXPORTATION EXCEL
# ==============================================================================
# Fréquences de consommation des 10 groupes (%)
freq_wdds_zone = df_wdds.groupby(col_zone)[labels_wdds_courts].mean() * 100
freq_wdds_global = df_wdds[labels_wdds_courts].mean() * 100

df_freq_wdds = freq_wdds_zone.T
df_freq_wdds['Global'] = freq_wdds_global
df_freq_wdds = df_freq_wdds.reset_index().rename(columns={'index': 'Groupe_Alimentaire_WDDS'})

# Synthèse du score moyen WDDS
stats_wdds = df.groupby(col_zone)['wdds_score'].agg(
    Moyenne='mean', Écart_Type='std', Médiane='median', Min='min', Max='max'
).reset_index()

stats_wdds_global = pd.DataFrame({
    col_zone: ['Global'],
    'Moyenne': [df['wdds_score'].mean()],
    'Écart_Type': [df['wdds_score'].std()],
    'Médiane': [df['wdds_score'].median()],
    'Min': [df['wdds_score'].min()],
    'Max': [df['wdds_score'].max()]
})
df_summary_wdds = pd.concat([stats_wdds, stats_wdds_global], ignore_index=True)

# Taux de couverture du MDD-W (%)
cross_mddw = pd.crosstab(df[col_zone], df['mddw_atteint'], normalize='index') * 100
cross_mddw_global = df['mddw_atteint'].value_counts(normalize=True) * 100
cross_mddw.loc['Global'] = cross_mddw_global
cross_mddw = cross_mddw.reset_index()

# Exportation Excel
fichier_excel_wdds = "Tableau_6_4_2_WDDS_MDDW_Performances.xlsx"
with pd.ExcelWriter(fichier_excel_wdds, engine='openpyxl') as writer:
    df_freq_wdds.to_excel(writer, sheet_name='Freq_10_Groupes', index=False)
    df_summary_wdds.to_excel(writer, sheet_name='Stats_Score_WDDS', index=False)
    cross_mddw.to_excel(writer, sheet_name='Taux_MDD_W', index=False)

print(f"--> Fichier Excel WDDS généré : {fichier_excel_wdds}")

# ==============================================================================
# 4. GRAPHIQUE BARPLOT COMPARATIF PAR ZONE (FIG 6.7)
# ==============================================================================
plt.close('all')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [2, 1]})

# Graphique 1 : Fréquences des 10 groupes d'aliments par zone
df_plot = df_freq_wdds.melt(id_vars=['Groupe_Alimentaire_WDDS'], 
                            value_vars=[c for c in df_freq_wdds.columns if c not in ['Groupe_Alimentaire_WDDS', 'Global']],
                            var_name='Zone', value_name='Pourcentage')

sns.barplot(data=df_plot, x='Pourcentage', y='Groupe_Alimentaire_WDDS', hue='Zone', palette='Set2', ax=ax1)
ax1.set_title("A. Consommation des 10 groupes d'aliments de la femme (%)", fontsize=11, fontweight='bold')
ax1.set_xlabel("Proportion de femmes (%)", fontsize=10)
ax1.set_ylabel("")
ax1.set_xlim(0, 100)

for p in ax1.patches:
    width = p.get_width()
    if width > 0:
        ax1.annotate(f"{width:.1f}%", (width + 1, p.get_y() + p.get_height() / 2.),
                     ha='left', va='center', fontsize=8)

# Graphique 2 : Taux d'atteinte du seuil MDD-W (>= 5)
df_mddw_plot = df.groupby(col_zone)['mddw_atteint'].apply(lambda x: (x == 'MDD-W Atteint (≥5)').mean() * 100).reset_index()
sns.barplot(data=df_mddw_plot, x=col_zone, y='mddw_atteint', palette='Set2', ax=ax2)
ax2.set_title("B. Proportion de femmes atteignant le MDD-W (≥5 groupes)", fontsize=11, fontweight='bold')
ax2.set_ylabel("Taux d'atteinte (%)", fontsize=10)
ax2.set_xlabel("Zone d'étude", fontsize=10)
ax2.set_ylim(0, 100)

for p in ax2.patches:
    height = p.get_height()
    ax2.annotate(f"{height:.1f}%", (p.get_x() + p.get_width() / 2., height / 2),
                 ha='center', va='center', fontsize=11, fontweight='bold', color='white')

plt.suptitle(f"Fig 6.7 : Diversité Alimentaire des Femmes (WDDS / MDD-W) (N = {len(df)})", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()

fichier_image_wdds = "Fig_6_7_WDDS_MDDW_Zone.png"
plt.savefig(fichier_image_wdds, dpi=300, bbox_inches='tight')
plt.show()

print(f"--> Graphique sauvegardé : {fichier_image_wdds}")

### 6.4.3 Disparités spatiales et croisements avec la typologie des exploitations

#### Objectif académique : Analyser le couplage/découplage entre la performance agroécologique des exploitations (Clusters) et les résultats nutritionnels (HDDS et WDDS).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ==============================================================================
# 1. PARAMÉTRAGE DES COLONNES
# ==============================================================================
col_cluster = 'Cluster_Label'  # Variable créée à la section 3 (CAH)
col_zone = 'zone'              # Modifiez en 'Zone' si votre colonne prend une majuscule

# Alignement de la variable HDDS brut
if 'hdds_score_brut' not in df.columns and 'hdds_score' in df.columns:
    df['hdds_score_brut'] = df['hdds_score']

# Ordre d'affichage des typologies
ordre_clusters = [
    'Type 1 : Extensifs Vivriers',
    'Type 2 : Maraîchers Agroécologiques',
    'Type 3 : Profils Autoconsommation'
]
clusters_presents = [c for c in ordre_clusters if c in df[col_cluster].unique()]

# ==============================================================================
# 2. STATISTIQUES DESCRIPTIVES CROISÉES (TYPOLOGIE x ZONE x NUTRITION)
# ==============================================================================
# Synthèse HDDS par Cluster et Zone
stats_hdds_cluster = df.groupby([col_cluster, col_zone])['hdds_score_brut'].agg(
    Effectif='count',
    Moyenne='mean',
    Écart_Type='std',
    Médiane='median',
    Min='min',
    Max='max'
).reset_index()

# Synthèse WDDS par Cluster et Zone
stats_wdds_cluster = df.groupby([col_cluster, col_zone])['wdds_score'].agg(
    Effectif='count',
    Moyenne='mean',
    Écart_Type='std',
    Médiane='median',
    Min='min',
    Max='max'
).reset_index()

# Croisement MDD-W (Proportions en %)
cross_mddw_cluster = pd.crosstab([df[col_cluster], df[col_zone]], df['mddw_atteint'], normalize='index') * 100
cross_mddw_cluster = cross_mddw_cluster.reset_index()

# ==============================================================================
# 3. TESTS STATISTIQUES (ANOVA, KRUSKAL-WALLIS & CHI-DEUX)
# ==============================================================================
hdds_groups = [group['hdds_score_brut'].dropna().values for _, group in df.groupby(col_cluster)]
wdds_groups = [group['wdds_score'].dropna().values for _, group in df.groupby(col_cluster)]

f_hdds, p_anova_hdds = stats.f_oneway(*hdds_groups)
h_hdds, p_kw_hdds = stats.kruskal(*hdds_groups)

f_wdds, p_anova_wdds = stats.f_oneway(*wdds_groups)
h_wdds, p_kw_wdds = stats.kruskal(*wdds_groups)

contingency_mddw = pd.crosstab(df[col_cluster], df['mddw_atteint'])
chi2_mddw, p_chi2_mddw, dof_mddw, _ = stats.chi2_contingency(contingency_mddw)

df_tests = pd.DataFrame({
    'Indicateur_Nutritionnel': ['Score HDDS (Ménage)', 'Score WDDS (Femme)', 'Seuil MDD-W (≥5)'],
    'Test_Utilise': ['ANOVA (F) / Kruskal-Wallis (H)', 'ANOVA (F) / Kruskal-Wallis (H)', 'Chi-deux (χ²)'],
    'Valeur_Statistique': [f'F={f_hdds:.2f} | H={h_hdds:.2f}', f'F={f_wdds:.2f} | H={h_wdds:.2f}', f'χ²={chi2_mddw:.2f}'],
    'p_value': [p_anova_hdds, p_anova_wdds, p_chi2_mddw],
    'Significatif_p_005': [p_anova_hdds < 0.05, p_anova_wdds < 0.05, p_chi2_mddw < 0.05]
})

# ==============================================================================
# 4. EXPORTATION EXCEL (TABLEAU 6.4.3)
# ==============================================================================
fichier_excel_643 = "Tableau_6_4_3_Croisement_Typologie_Nutrition.xlsx"
with pd.ExcelWriter(fichier_excel_643, engine='openpyxl') as writer:
    stats_hdds_cluster.to_excel(writer, sheet_name='HDDS_par_Cluster', index=False)
    stats_wdds_cluster.to_excel(writer, sheet_name='WDDS_par_Cluster', index=False)
    cross_mddw_cluster.to_excel(writer, sheet_name='MDDW_par_Cluster', index=False)
    df_tests.to_excel(writer, sheet_name='Tests_Statistiques', index=False)

# ==============================================================================
# 5. GÉNÉRATION DE LA FIGURE 6.8 (BOXPLOTS)
# ==============================================================================
plt.close('all')
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.boxplot(data=df, x=col_cluster, y='hdds_score_brut', hue=col_zone, palette='Set2', order=clusters_presents, ax=axes[0])
axes[0].set_title("A. Score HDDS par Typologie d'Exploitation et Zone", fontweight='bold')
axes[0].set_xlabel("Typologie des Exploitations")
axes[0].set_ylabel("Score HDDS (0-12)")
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

sns.boxplot(data=df, x=col_cluster, y='wdds_score', hue=col_zone, palette='Set2', order=clusters_presents, ax=axes[1])
axes[1].set_title("B. Score WDDS par Typologie d'Exploitation et Zone", fontweight='bold')
axes[1].set_xlabel("Typologie des Exploitations")
axes[1].set_ylabel("Score WDDS (0-10)")
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle(f"Fig 6.8 : Croisement Typologie Agroécologique x Scores Nutritionnels (N = {len(df)})", 
             fontsize=12, fontweight='bold', y=1.03)

plt.tight_layout()
fichier_image_643 = "Fig_6_8_Croisement_Cluster_Nutrition.png"
plt.savefig(fichier_image_643, dpi=300, bbox_inches='tight')
plt.show()

print(f"--> Figure sauvegardée : {fichier_image_643}")
print(f"--> Tableau Excel sauvegardé : {fichier_excel_643}")